In [ ]:
import time
import random

def run_kmer_simulation(n_ratio, k, test_lambda, duration=30):
    """
    Simulates a DNA stream with specific bit encodings and tests a bitwise lambda.

    Encodings: N=1, A=12, C=13, G=14, T=15
    """
    # Define the bit mappings
    N_BIT, A_BIT, C_BIT, G_BIT, T_BIT = 1, 12, 13, 14, 15

    # Probability distribution
    p_n = n_ratio
    p_others = (1.0 - p_n) / 4.0

    population = [A_BIT, C_BIT, G_BIT, T_BIT, N_BIT]
    weights = [p_others, p_others, p_others, p_others, p_n]

    # Mask to keep only the lower 4*k bits (the k-mer window)
    mask = (1 << (4 * k)) - 1

    samp_hits = 0
    sig_hits = 0
    total_tests = 0

    no_n_mask = 0

    for i in range(k):
      no_n_mask <<= 4
      no_n_mask += 0xC

    def test_for_n(w):
        # if 1st 2 bits are 0, means n
        return not (~w & no_n_mask)

    # Initialize a window with random nucleotides
    current_window = 0
    for _ in range(k):
        new_nibble = random.choices(population, weights=weights)[0]
        current_window = ((current_window << 4) | new_nibble) & mask

    print(f"Starting simulation on {test_lambda.__name__}, p = {n_ratio}, for {duration}s...")
    start_time = time.time()

    # Loop for the specified duration
    while time.time() - start_time < duration:
        # Generate next character
        new_nibble = random.choices(population, weights=weights)[0]

        # Slide the window (shift left 4 bits, add new, mask for k-mer size)
        current_window = ((current_window << 4) | new_nibble) & mask

        # Test the lambda
        if test_lambda(current_window):
            samp_hits += 1
            if test_for_n(current_window):
                sig_hits += 1

        total_tests += 1

    # Final statistics
    sig_probability = (sig_hits / total_tests) if total_tests > 0 else 0
    samp_probability = (samp_hits / total_tests) if total_tests > 0 else 0
    warp_div_estimate = 1 - (1 - samp_probability)**32
    single_hit_estimate = 1 - (1 - sig_probability)**(3000 - max(24, k) + 1)
    speedup_estimate = 1000 / ((1 - warp_div_estimate) * (1 + (1 - single_hit_estimate) * 1000) + warp_div_estimate * 1001)

    print("\n--- Results ---")
    print(f"Total Windows Tested: {total_tests:,}")
    print(f"Total Sig Hits:           {sig_hits:,}")
    print(f"Total Samp Hits:           {samp_hits:,}")
    print(f"Sig Hit Probability:  {sig_probability:.6f} (~1/{int(1/sig_probability) if sig_probability > 0 else 0})")
    print(f"Samp Hit Probability:  {samp_probability:.6f} (~1/{int(1/samp_probability) if samp_probability > 0 else 0})")
    print(f"Est. Warp Divergence: {warp_div_estimate:.2%} (for 32 threads)")
    print(f"Prob in Singature (len 3k): {single_hit_estimate:.2%}")
    print(f"Est. Speedup: {speedup_estimate:.2%}\n")


def parity_8mer_lambda(w):
    # all 8 bits must be C/T/N
    return not ((w & 0x11111111) ^ 0x11111111)

def parity_8mer_lambda_xtra(w):
    # all 8 bits must be C/T/N, last bit must be C/N
    return not ((w & 0x11111111) ^ 0x11111111) | ((w & 0x3) ^ 0x1)

